# Import dataset

In [ ]:
import sys
print(sys.executable)

In [ ]:
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns 
import numpy as np
import json

In [ ]:
# read bigquery data into pandas dataframe
import pandas as pd

df = pd.read_gbq(
    """
    SELECT  *
    FROM `jr-data-training.cafe.cafe-sales`
    --- LIMIT 10
  """,
    project_id="jr-data-training",
    location="australia-southeast1",
)

# Evaluation of the average revenue per customer


Using RFM analysis to divide customers into different segments:

- Recency: How recently has the customer made a transaction with us
- Frequency: How frequent is the customer in ordering/buying some product from us
- Monetary: How much does the customer spend on purchasing products from us.

In [ ]:
# Create a RFM table
rfm_df = df.groupby('customer_id').agg(
    Recency=('date_created', lambda x: (df['date_created'].max() - x.max()).days),
    Frequency=('date_created', 'count'),
    Monetary=('total', 'sum')
).reset_index()

## Data Pre-processing

In [ ]:
fig = px.histogram(rfm_df, x='Recency', title='Recency Distribution', nbins=20, histnorm='probability density')
fig.show()

fig = px.histogram(rfm_df, x='Frequency', title='Frequency Distribution', nbins=20, histnorm='probability density')
fig.show()

fig = px.histogram(rfm_df, x='Monetary', title='Monetary Value Distribution', nbins=20, histnorm='probability density')
fig.show()

Based on the graph above, data transformation is needed as all of them are right-skewed form. 

### Data transformation

#### Initial transformation test

In [ ]:
from scipy import stats

'''
 Function for experimenting different transformation methods:

    - log transformation
    - square root transformation
    - box-cox transformation Note
    
'''

def analyze_skewness(x):

    fig, ax = plt.subplots(2, 2, figsize=(6, 6))

    # To ensure the data is positive / non-zero
    adjust_x = np.clip(rfm_df[x], 1e-6, None)

    sns.histplot(adjust_x, ax=ax[0, 0], kde=True)
    sns.histplot(np.log(adjust_x), ax=ax[0, 1], kde=True)
    sns.histplot(np.sqrt(adjust_x), ax=ax[1, 0], kde=True)
    sns.histplot(stats.boxcox(adjust_x)[0], ax=ax[1, 1], kde=True)

    plt.tight_layout()
    plt.show()

    skew_coe = adjust_x.skew().round(2)

    print('Log Transform : The skew coefficient of', skew_coe, 'to', np.log(skew_coe))
    print('Square Root Transform : The skew coefficient of', skew_coe, 'to', np.sqrt(skew_coe))
    print('Box-Cox Transform : The skew coefficient of', skew_coe, 'to', pd.Series(stats.boxcox(adjust_x)[0]).skew().round(2))


In [ ]:
analyze_skewness('Recency')

In [ ]:
analyze_skewness('Frequency')

In [ ]:
analyze_skewness('Monetary')

The graphs and calculation result showed that the variables with **box-cox transformation** shows a more symmetrical form rather than the other transformations.

#### Adjust the data with Box-Cox Transform

In [ ]:
rfm_transformed = pd.DataFrame()

rfm_transformed["R_transformed"] = pd.Series(np.cbrt(rfm_df['Recency'])).values
rfm_transformed["F_transformed"] = stats.boxcox(rfm_df['Frequency'])[0]
rfm_transformed["M_transformed"] = stats.boxcox(rfm_df['Monetary'])[0]

Since the columns dont have the same mean and variance, we have to normalize it using the StandardScaler.

In [ ]:
from sklearn.preprocessing import StandardScaler


scaler = StandardScaler()

# Fit and Transform The Data
scaler.fit(rfm_transformed)
rfm_normalized = scaler.transform(rfm_transformed)

rfm_normalized = pd.DataFrame(rfm_normalized)

## Data segmentation (K-means)

In [ ]:
plt.figure(figsize=(12,8))

from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist
K = range(1, 20)

distortions = []
inertias = []
mapping1 = {}
sse = {}

for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(rfm_normalized)
    
    distortions.append(sum(np.min(cdist(rfm_normalized, kmeans.cluster_centers_,
                                        'euclidean'), axis=1)) / rfm_normalized.shape[0])
    inertias.append(kmeans.inertia_)
 
    mapping1[k] = sum(np.min(cdist(rfm_normalized, kmeans.cluster_centers_,
                                   'euclidean'), axis=1)) / rfm_normalized.shape[0]
    
    sse[k] = kmeans.inertia_ # SSE to closest cluster centroid

plt.title('The Elbow Method')
plt.xlabel('k')
plt.ylabel('SSE')
sns.pointplot(x=list(sse.keys()), y=list(sse.values()))
plt.show()

In [ ]:
for key, val in mapping1.items():
    print(f'{key} : {val}')

We are going to choose the optimal parameter by observing where the k-value will exhibit a linear trend on the following successive k. Based on the plot above, the k-value of **3 or 4** is the ideal hyperparameter for the model as the following k-value tends to have a linear relationship. 

### Comparsion between 3/4 cluster

In [ ]:
model = KMeans(n_clusters=3, random_state=42)
model.fit(rfm_normalized)

rfm_df["Cluster_group"] = model.labels_
rfm_df.groupby('Cluster_group').agg({
    'Recency':'mean',
    'Frequency':'mean',
    'Monetary':['mean', 'count']}).round(1)

In [ ]:
model = KMeans(n_clusters=4, random_state=42)
model.fit(rfm_normalized)

rfm_df["Cluster_group"] = model.labels_
rfm_df.groupby('Cluster_group').agg({
    'Recency':'mean',
    'Frequency':'mean',
    'Monetary':['mean', 'count']}).round(1)

### Fit the K-means model